# SPK-9 — Shuffle internals & stages

**Break -> Detect -> Fix -> Prove.** Every Spark job is a chain of **stages** separated by
**shuffles**. This notebook makes the invisible visible: first it shows *where* a shuffle
boundary comes from (narrow vs wide dependencies, read off `df.explain()`), then it sweeps the
post-shuffle **partition count** across three values to prove that too few partitions **spill**
and too many drown in **scheduling overhead** -- with a sweet spot in between.

**Pre-requisite:** the unified Spark server is up (`make up`). This notebook connects via Spark
Connect. **Open the Spark UI at http://localhost:4040** and watch it while the cells run.

**Laptop-safe:** ~20M rows are *generated lazily* and only `count()`-ed -- never collected or
written -- so nothing fills memory or disk. This is a partition-sizing demo, not a memory bomb,
so the default **tuned** box is fine (no need for `make up-constrained`). Nothing to delete at the end.

See the companion writeup in [`README.md`](./README.md), the
[Spark-UI guide](../../docs/spark-ui-guide.md), and the [troubleshooting sheet](../../docs/troubleshooting.md).

In [ ]:
from common.spark_session import spark, display_df
from common.profiles import apply_profile, profile_summary
from common.datagen import uniform_keys
from common.metrics_diff import measure, compare
from pyspark.sql import functions as F

# Watch this while the cells run -- shuffles show up as `Exchange` in the SQL tab,
# and each reduce stage runs exactly `spark.sql.shuffle.partitions` tasks.
print("Spark UI:", "http://localhost:4040")
spark

## Step 0 -- Parameters & a balanced dataset

We synthesize a **balanced** `events` fact with `common.datagen.uniform_keys` (no skew -- that's
[`SPK-1`](../skew/README.md)). Keeping the data balanced means the **partition count** is the
*only* variable in the sweep below, so the runtime differences are purely about task size vs
task count -- nothing else.

In [ ]:
# Lower N_ROWS to 5-10M if your laptop is slow; raise it to make spill/overhead more dramatic.
N_ROWS = 20_000_000
N_KEYS = 2_000          # group-by cardinality: enough groups to be a real aggregation, evenly spread

events = uniform_keys(spark, n_rows=N_ROWS, n_keys=N_KEYS, key_col="key")
print(f"events: ~{N_ROWS:,} rows over {N_KEYS:,} evenly-spread keys (lazy -- nothing stored yet)")
display_df(events.limit(5))

## 1. Narrow vs wide dependencies -- who creates a shuffle?

A **narrow** transformation needs no data movement: each output partition is computed from exactly
one input partition. `select`, `filter`, and `withColumn` are narrow -- they **fuse into one
stage** with **no shuffle**.

A **wide** transformation must redistribute rows across the network so matching keys land together.
`groupBy`, `distinct`, and `join` are wide -- each inserts an **`Exchange`** node (= a shuffle =
a **stage boundary**).

The way to *see* this is `df.explain()`: **find the `Exchange` node(s).** (Connect-safe: we read
the plan only -- never touch the RDD or `getNumPartitions`.)

In [ ]:
# NARROW chain: select -> filter -> withColumn. Expect NO `Exchange` in the plan -> one stage.
narrow = (events
          .select("row_id", "key", "amount")
          .filter(F.col("amount") > 50.0)
          .withColumn("amount_tax", F.col("amount") * 1.1))

print("NARROW (select / filter / withColumn) -- expect ZERO Exchange nodes, ONE stage:\n")
narrow.explain()

In [ ]:
# WIDE: a groupBy aggregation. Expect exactly ONE `Exchange` (hashpartitioning) -> a new stage.
# That Exchange is the shuffle boundary: the map side writes shuffle files, the reduce side reads them.
wide = events.groupBy("key").agg(F.sum("amount").alias("total"))

print("WIDE (groupBy.agg) -- expect ONE Exchange node = the shuffle / stage boundary:\n")
wide.explain()

## 2. Break it -- the partition-count sweep

Now the core experiment. We run the **same ~20M-row aggregation** three times, changing only
`spark.sql.shuffle.partitions`. We hold the **`constrained`** profile (AQE **off**) so the count
we set is the count that actually runs -- no runtime coalescing masking the effect.

The reduce stage runs **exactly** `shuffle.partitions` tasks, so this knob sets each task's slice
of the data:

- **8** -> the 20M shuffled rows pack into 8 huge partitions -> 8 fat tasks that **spill** memory->disk.
- **200** -> Spark's default; here it's the **sweet spot** (task size and count both reasonable).
- **2000** -> 2000 tiny partitions -> **scheduling overhead** dwarfs the actual compute.

Watch the **Stages** tab: the reduce stage's task count changes to match each setting.

In [ ]:
# Baseline: AQE off so `shuffle.partitions` is authoritative (no runtime coalesce).
apply_profile(spark, "constrained")
profile_summary(spark)

# The aggregation under test -- identical across all three runs (only the conf changes).
def agg():
    return events.groupBy("key").agg(F.sum("amount").alias("total")).count()

In [ ]:
# Run the SAME aggregation at three partition counts (only the conf changes between runs).
# Watch the Stages tab: the reduce stage's task count tracks each setting.
#   8    -> few fat tasks   -> non-zero Spill (disk), slow
#   200  -> Spark's default -> little/no spill, fastest (the sweet spot)
#   2000 -> tiny tasks      -> no spill but scheduling overhead, slow again
sweep = {}
for parts in (8, 200, 2000):
    apply_profile(spark, "constrained", **{"spark.sql.shuffle.partitions": str(parts)})
    sweep[parts] = measure(spark, f"parts={parts}", agg)
    print(f"parts={parts} metrics:", sweep[parts])

p8, p200, p2000 = sweep[8], sweep[200], sweep[2000]

## 3. Detect it -- read the Spark UI

http://localhost:4040 -> **SQL / DataFrame** -> click a query -> its plan DAG shows the single
`Exchange` (the shuffle). Then -> **Stages** -> the reduce stage. The tells (see
[`docs/spark-ui-guide.md`](../../docs/spark-ui-guide.md)):

- **Number of tasks** == `spark.sql.shuffle.partitions` (8, ~200, 2000 respectively).
- **parts=8:** non-zero **Spill (disk)** in Summary Metrics -- partitions too big to fit in memory.
- **parts=2000:** thousands of tiny tasks; a large **Scheduler Delay** fraction in the task
  Event Timeline -- overhead, not compute.

The cell below prints the same story as numbers straight from the sweep. (Connect-safe: task
count comes from the metrics REST API, not `rdd.getNumPartitions()`.)

In [ ]:
def line(m):
    print(f"  {m['label']:<11} runtime={m['runtime_s']:>6} s  "
          f"tasks={str(m['num_tasks']):>5}  "
          f"spill_disk={m['spill_disk_bytes']}  "
          f"task_max_ms={m['task_time_max_ms']}")

print("partition-count sweep (balanced data -- count is the only variable):")
for m in (p8, p200, p2000):
    line(m)
print("\nReduce-stage task count == spark.sql.shuffle.partitions; 8 spills, 2000 adds overhead.")

## 4. Prove it

Before/after across the sweep. Look for the **U-shape in Wall-clock runtime** (slow at 8 from
spill, fast at 200, slow again at 2000 from overhead), the **Tasks** row tracking the partition
count, and **Spill (disk)** non-zero only at 8.

In [ ]:
compare([p8, p200, p2000])

## 5. Fix it & tie-back

**Right-size the count.** Rule of thumb: **~2-3x total executor cores** (here, the local box's
cores). On this laptop the default `200` already lands near the sweet spot. The modern answer is
to **let AQE coalesce**: set `spark.sql.adaptive.enabled=true` +
`adaptive.coalescePartitions.enabled=true` (the `tuned` profile), start with a generous
`shuffle.partitions`, and AQE **merges** small post-shuffle partitions at runtime so you don't
have to guess ([`SPK-6`](../README.md) AQE deep-dive).

**How this connects to the rest of Phase 1:**
- **Too few partitions -> spill.** The `parts=8` run is the disk-spill pathology of [`SPK-4`](../spill/README.md) in miniature.
- **Skew -> one fat partition.** Even with a good count, a single hot key floods **one** reduce partition -> a straggler the count can't fix: [`SPK-1`](../skew/README.md).
- **AQE auto-tunes this.** [`SPK-6`](../README.md) coalesces tiny partitions (and splits skewed ones) at runtime.

## Takeaways & "in real production..."

- A **wide dependency** (`groupBy` / `join` / `distinct`) inserts an **`Exchange`** = a shuffle =
  a **new stage**; **narrow** ops (`select` / `filter` / `withColumn`) don't. Read `df.explain()`
  and **find the `Exchange` nodes** -- each is a cost and a stage boundary.
- `spark.sql.shuffle.partitions` = the reduce stage's **task count**, which sets **task size**.
  **Too few -> spill; too many -> scheduling overhead.** Aim for ~**2-3x total cores**, or let AQE coalesce.
- **In prod:** keep **AQE enabled**, don't copy a giant `shuffle.partitions` from someone else's
  TB-scale job, and when a stage is mysteriously slow check **task count** and **spill** before
  reaching for more hardware.

## Teardown

Nothing was written (we only counted generated data), so there is nothing to delete. We just
restore the production-tuned safety nets.

In [ ]:
apply_profile(spark, "tuned")        # restore production-tuned safety nets (AQE on, default 200)
spark.catalog.clearCache()
print("Done. Profile reset to 'tuned'. No tables/files were created; `make clean` clears .tmp if needed.")